# Cross-validation sweep — Bangla multimodal political bias

Runs every model in `configs/` under the **one** protocol the revised paper
reports: grouped stratified 5-fold CV, augmentation applied inside each training
fold, out-of-fold predictions pooled.

Use a GPU runtime (**Runtime -> Change runtime type -> T4 GPU**). The sweep does
not train on CPU in reasonable time, and Apple MPS hangs on these encoders.

Each model writes `experiments/cv-<name>-<timestamp>/` with `cv.json`
(per-fold scores, pooled score, bootstrap interval, population, git commit) and
`predictions.csv`. Download `experiments/` at the end and the local
`bmpb leaderboard` builds the results table from it.

In [ ]:
!nvidia-smi -L
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. The repository

The repo is private, so cloning needs a token. In Colab: key icon in the left
sidebar -> add a secret named `GH_TOKEN` holding a GitHub personal access token
with `repo` scope.

In [ ]:
from google.colab import userdata
import subprocess, os

token = userdata.get("GH_TOKEN")
url = f"https://{token}@github.com/kishormorol/bangla-multimodal-political-bias.git"
if not os.path.exists("bangla-multimodal-political-bias"):
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
%cd bangla-multimodal-political-bias
!git log --oneline -1

In [ ]:
!pip install -q -e . 2>&1 | tail -2
!pip install -q gdown 2>&1 | tail -1

## 2. The data

`bmpb data` mirrors the Drive folder into `data/raw/`. Google throttles bulk
folder downloads, so if the image counts come back short, run the cell again —
it resumes and only fetches what is missing.

In [ ]:
!python -m bmpb.cli data

### Article bodies

`bmpb backfill` refetches each item's source URL and pulls the article body out
of the page, so the text models see articles rather than 8-word headlines. It
recovers roughly two thirds of the corpus; the misses are outlets that refuse
scripted requests. `ingest` records `text_level` per item so the two populations
can be reported separately.

In [ ]:
!python -m bmpb.cli backfill --delay 1.0

In [ ]:
!python -m bmpb.cli ingest
!python -m bmpb.cli audit

## 3. The sweep

Order matters only for your patience: the text encoders are quick, the
vision-language models are not (BLIP and ViLT run at 384px). Run the text cell
first so there are numbers in hand early.

In [ ]:
!bash scripts/run_cv.sh configs/text

In [ ]:
!bash scripts/run_cv.sh configs/multimodal

## 4. Results

`leaderboard` puts the cross-validated rows in one table and keeps the originally
published numbers separate, since those came from three different protocols and
are not comparable with these.

In [ ]:
!python -m bmpb.cli leaderboard
print(open("reports/tables/leaderboard.md").read())

## 5. Take the runs home

`experiments/` and `reports/` are what the paper's tables are built from. Commit
them back, or download the archive and unpack it into the local checkout.

In [ ]:
!tar czf cv-runs.tar.gz experiments reports
from google.colab import files
files.download("cv-runs.tar.gz")